# Colab 常规数据集训练（ramanv2）

此 notebook 用于常规 profile 的数据恢复、训练、验证集评估、解释性分析和结果打包。

In [ ]:
from pathlib import Path
import shutil
import sys
import tempfile
import zipfile

PROJECT_ROOT = Path("/content")
PACKAGE_ARCHIVE = PROJECT_ROOT / "ramanv2.zip"
PACKAGE_DIR = PROJECT_ROOT / "ramanv2"
PACKAGE_PARENT = PACKAGE_DIR.parent


def load_package():
    """检测到压缩包时替换 ramanv2；否则使用已解压的包。"""
    if PACKAGE_ARCHIVE.is_file():
        temp_dir = Path(tempfile.mkdtemp(prefix="ramanv2_unpack_", dir=PROJECT_ROOT))
        try:
            with zipfile.ZipFile(PACKAGE_ARCHIVE) as archive:
                archive.extractall(temp_dir)
            unpacked_dir = temp_dir / PACKAGE_DIR.name
            if not unpacked_dir.is_dir():
                raise FileNotFoundError("压缩文件中未找到 ramanv2/ 目录")
            if PACKAGE_DIR.exists():
                if not PACKAGE_DIR.is_dir():
                    raise FileExistsError(f"包路径不是目录：{PACKAGE_DIR}")
                shutil.rmtree(PACKAGE_DIR)
            shutil.move(str(unpacked_dir), str(PACKAGE_PARENT))
            print(f"replaced package: {PACKAGE_ARCHIVE.name} -> {PACKAGE_DIR}")
        finally:
            shutil.rmtree(temp_dir, ignore_errors=True)
    elif not PACKAGE_DIR.is_dir():
        raise FileNotFoundError("请先将 ramanv2.zip 上传到 /content")
    if str(PACKAGE_PARENT) not in sys.path:
        sys.path.insert(0, str(PACKAGE_PARENT))


load_package()


In [ ]:
import importlib
import sys


def reload_ramanv2():
    """清除 ramanv2 模块缓存，以加载当前目录中的最新代码。"""
    for name in list(sys.modules):
        if name == "ramanv2" or name.startswith("ramanv2."):
            del sys.modules[name]
    importlib.invalidate_caches()
    config_module = importlib.import_module("ramanv2.core.config")
    print("ramanv2 modules reloaded.")
    return config_module


config_module = reload_ramanv2()
build_config = config_module.build_config


In [ ]:
from ramanv2.core.config import build_config
from ramanv2.data.profiles import get_dataset_dir, get_profile

# 常规数据集 profile；使用 ASCII 标识，例如 GN、MICRO、ENT、NENT、GP、FUNG。
PROFILE_ID = "GN"
LEVEL_NAME = "level_1"
PARENT_NAME = None
PARENT_INDEX = None
# 填写整数时覆盖本次运行的随机种子；None 时使用 config.py 的默认值。
SEED = None
# None 时自动创建 output/<profile>/<时间戳>；填写路径时使用指定实验目录。
EXPERIMENT_DIR = None
LAST_TRAINED_RUN_DIR = None

profile = get_profile(PROFILE_ID)
dataset_dir = get_dataset_dir(profile, PROJECT_ROOT)
config_values = {"profile_id": PROFILE_ID}
if SEED is not None:
    config_values["seed"] = int(SEED)
config = build_config(config_values)
print(dataset_dir)


## 恢复 init.npz

In [ ]:
from datetime import datetime
from uuid import uuid4

from ramanv2.data.io import unpack_init

INIT_ARCHIVE = dataset_dir / profile.root_init_pack
INIT_DIR = dataset_dir / profile.root_init
UNPACK_INIT_ENABLE = True


def replace_init_from_archive(archive_path, init_dir):
    """将 archive 先恢复到临时目录，成功后替换当前 init 目录。"""
    stage_dir = init_dir.with_name(f".{init_dir.name}_unpack_{uuid4().hex}")
    unpack_init(archive_path, stage_dir)
    if init_dir.exists():
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_dir = init_dir.with_name(f"{init_dir.name}_backup_{timestamp}")
        init_dir.rename(backup_dir)
        print(f"previous_init_backup = {backup_dir}")
    stage_dir.rename(init_dir)
    print(f"restored_init_dir = {init_dir}")


if UNPACK_INIT_ENABLE:
    replace_init_from_archive(INIT_ARCHIVE, INIT_DIR)


## 构建训练集

In [ ]:
from ramanv2.data.build import build_train

# build_train 会在成功后以新 train/ 替换当前产物，并保留同级备份目录。
build_train(profile, dataset_dir, input_config=config.input)


In [ ]:
from ramanv2.data.count import count_dataset, print_count_results

tree, sample_count = count_dataset(dataset_dir / profile.root_train_clean)
print_count_results(tree, sample_count)


## 训练

In [ ]:
from dataclasses import asdict

# 训练配置总览
def print_config_summary(config):
    for group_name, values in asdict(config).items():
        print(f"\n===== {group_name} =====")
        for key, value in values.items():
            print(f"  {key}: {value}")


print_config_summary(config)
print("\n===== training_request =====")
print(f"  level_name: {LEVEL_NAME}")
print(f"  parent_name: {PARENT_NAME}")
print(f"  parent_index: {PARENT_INDEX}")


In [ ]:
from datetime import datetime

from ramanv2.training.workflow import TrainRequest, run_training


if EXPERIMENT_DIR is None:
    active_experiment_dir = (
        PROJECT_ROOT
        / "output"
        / profile.dataset_name
        / datetime.now().strftime("%Y%m%d_%H%M%S")
    )
else:
    active_experiment_dir = Path(EXPERIMENT_DIR)

request = TrainRequest(
    config=config,
    level_name=LEVEL_NAME,
    only_parent=PARENT_INDEX,
    only_parent_name=PARENT_NAME,
    experiment_dir=active_experiment_dir,
)
training_meta = run_training(request)
EXPERIMENT_DIR = active_experiment_dir

trained_run_dirs = [
    EXPERIMENT_DIR / entry["run_dir"]
    for entries in training_meta.get("runs", {}).values()
    for entry in entries
    if entry.get("status") == "trained" and entry.get("run_dir")
]
if len(trained_run_dirs) == 1:
    LAST_TRAINED_RUN_DIR = trained_run_dirs[0]
    print(f"last_trained_run_dir = {LAST_TRAINED_RUN_DIR}")
elif trained_run_dirs:
    print("本次训练生成多个 run_* 目录；请在单模型检查中手动选择。")
else:
    print("本次训练未生成可用于单模型检查的 run_* 目录。")

print(f"experiment_dir = {EXPERIMENT_DIR}")


## 第二阶段微调（可选）

In [ ]:
# 从刚完成训练的最佳模型权重启动第二阶段微调。
# 这是新的优化过程：会重建优化器与学习率调度器，不等同于 checkpoint 精确续训。
SECOND_STAGE_ENABLE = False
SECOND_STAGE_EPOCHS = 40
SECOND_STAGE_LEARNING_RATE = 5e-5

if SECOND_STAGE_ENABLE:
    from dataclasses import replace
    from datetime import datetime

    import torch

    if LAST_TRAINED_RUN_DIR is None:
        raise ValueError("请先完成第一阶段训练，或手动填写 LAST_TRAINED_RUN_DIR。")
    first_stage_model_paths = sorted(Path(LAST_TRAINED_RUN_DIR).glob("*_model.pt"))
    if len(first_stage_model_paths) != 1:
        raise ValueError(f"第一阶段 run 中应有唯一模型文件，实际为：{first_stage_model_paths}")

    first_stage_model_path = first_stage_model_paths[0]
    first_stage_state = torch.load(first_stage_model_path, map_location="cpu", weights_only=True)

    def load_first_stage_weights(model, train_task):
        # 新分类模型的结构与第一阶段一致，严格加载其最佳权重。
        model.load_state_dict(first_stage_state, strict=True)

    second_stage_config = replace(
        config,
        training=replace(
            config.training,
            epochs=SECOND_STAGE_EPOCHS,
            learning_rate=SECOND_STAGE_LEARNING_RATE,
            scheduler_t_max=SECOND_STAGE_EPOCHS,
        ),
    )
    second_stage_run_name = f"run_stage2_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
    second_stage_meta = run_training(
        TrainRequest(
            config=second_stage_config,
            level_name=LEVEL_NAME,
            only_parent=PARENT_INDEX,
            only_parent_name=PARENT_NAME,
            experiment_dir=EXPERIMENT_DIR,
            run_name=second_stage_run_name,
            initialize_model=load_first_stage_weights,
        )
    )
    second_stage_run_dirs = sorted(EXPERIMENT_DIR.rglob(second_stage_run_name))
    if len(second_stage_run_dirs) != 1:
        raise RuntimeError(f"第二阶段应生成唯一 run，实际为：{second_stage_run_dirs}")
    LAST_TRAINED_RUN_DIR = second_stage_run_dirs[0]
    print(f"second_stage_source_model = {first_stage_model_path}")
    print(f"last_trained_run_dir = {LAST_TRAINED_RUN_DIR}")


## 单模型评估

In [ ]:
import re

from IPython.display import Image, display
from ramanv2.evaluation.baseline import BaselineSpec, evaluate_baseline_run
from ramanv2.evaluation.model_eval import evaluate_model_run

# None 时优先使用本 notebook 刚完成训练的唯一 run_* 目录。
# 查看其他实验时，填写具体 run_* 目录。
SINGLE_RUN_DIR = None
SINGLE_LEVEL = "level_1"
SINGLE_EVALUATE_ENABLE = True
CPU_ENABLE = False


def parse_validation_metric(report_text, metric_name):
    match = re.search(
        rf"^{re.escape(metric_name)}[ ]+([0-9.]+)%",
        report_text,
        flags=re.MULTILINE,
    )
    if match is None:
        raise ValueError(f"未在评估报告中找到 {metric_name}")
    return float(match.group(1))


def print_validation_metrics(result_name, result_dir, report_name):
    report_text = (result_dir / report_name).read_text(encoding="utf-8")
    accuracy = parse_validation_metric(report_text, "Accuracy")
    macro_f1 = parse_validation_metric(report_text, "Macro F1-score")
    macro_recall = parse_validation_metric(report_text, "Macro Recall")
    print(f"\n{result_name} 验证集指标")
    print("=====================================")
    print(f" Val Set Accuracy:  {accuracy:.4f}%")
    print(f" Macro F1-score:    {macro_f1:.4f}%")
    print(f" Macro Recall:      {macro_recall:.4f}%")
    print("=====================================")


resolved_single_run_dir = SINGLE_RUN_DIR or LAST_TRAINED_RUN_DIR
if SINGLE_EVALUATE_ENABLE:
    if resolved_single_run_dir is None:
        raise ValueError("请先训练一个唯一模型，或填写 SINGLE_RUN_DIR")
    device = "cpu" if CPU_ENABLE else None
    single_model_result_dir = evaluate_model_run(
        resolved_single_run_dir,
        SINGLE_LEVEL,
        device,
    )
    single_baseline_result_dir = evaluate_baseline_run(
        resolved_single_run_dir,
        SINGLE_LEVEL,
        BaselineSpec(),
    )
    print(single_model_result_dir)
    print(single_baseline_result_dir)
    for result_name, result_dir, report_name in (
        ("模型", single_model_result_dir, "classification_report.txt"),
        ("PCA-SVM 基线", single_baseline_result_dir, "metrics.txt"),
    ):
        figure_path = result_dir / "confusion_matrix.png"
        print_validation_metrics(result_name, result_dir, report_name)
        print(f"{result_name} 混淆矩阵：{figure_path}")
        display(Image(filename=str(figure_path)))


## 单模型 Analysis

In [ ]:
from IPython.display import Image, display
from ramanv2.analysis.runner import run_interpret_run

SINGLE_ANALYZE_ENABLE = True
CPU_ENABLE = False
# 归因只抽取最多 5 个 batch；IG 步数仍沿用 run 保存的 32。
ANALYSIS_ATTRIBUTION_BATCH_COUNT = 5

resolved_single_run_dir = SINGLE_RUN_DIR or LAST_TRAINED_RUN_DIR
if SINGLE_ANALYZE_ENABLE:
    if resolved_single_run_dir is None:
        raise ValueError("请先训练一个唯一模型，或填写 SINGLE_RUN_DIR")
    single_analysis_result_dir = run_interpret_run(
        resolved_single_run_dir,
        SINGLE_LEVEL,
        "cpu" if CPU_ENABLE else None,
        attribution_batch_count=ANALYSIS_ATTRIBUTION_BATCH_COUNT,
    )
    print(single_analysis_result_dir)
    figure_dir = single_analysis_result_dir / "figures"
    for figure_name in ("channel_importance_IG.png", "layer_importance.png"):
        figure_path = figure_dir / figure_name
        print(figure_path)
        display(Image(filename=str(figure_path)))
    analysis_log_path = single_analysis_result_dir / "logs" / "analysis_log.txt"
    if analysis_log_path.is_file():
        print("\n===== Analysis 摘要 =====")
        print(analysis_log_path.read_text(encoding="utf-8"))
    else:
        print("未生成 Analysis 摘要日志。")


## 级联评估

In [ ]:
EXPERIMENT_DIR = "" # 如果需要更换评估主体

In [ ]:
from ramanv2.evaluation.model_eval import evaluate_model_cascade

# 使用训练后或手动填写的 EXPERIMENT_DIR，展示端到端多层模型预测。
CASCADE_LEVEL = "level_2"
CASCADE_EVALUATE_ENABLE = True
CPU_ENABLE = False

if CASCADE_EVALUATE_ENABLE:
    if not EXPERIMENT_DIR:
        raise ValueError("请先填写 EXPERIMENT_DIR")
    cascade_result_dir = evaluate_model_cascade(
        EXPERIMENT_DIR,
        CASCADE_LEVEL,
        "cpu" if CPU_ENABLE else None,
    )
    print(cascade_result_dir)


### 下层模型分析（真实父类路由）

按样本真实父类选择下层模型，输出下层模型的归因汇总；唯一子类分支继承上级归因。


In [ ]:
from ramanv2.analysis.runner import run_interpret_parent_routed

PARENT_ROUTED_ANALYSIS_ENABLE = True
PARENT_ROUTED_PARENT = None  # None 表示分析该层全部父类；可填父类名称以限定一个分支。

if PARENT_ROUTED_ANALYSIS_ENABLE:
    if not EXPERIMENT_DIR:
        raise ValueError("请先填写 EXPERIMENT_DIR")
    parent_routed_analysis_dir = run_interpret_parent_routed(
        EXPERIMENT_DIR,
        CASCADE_LEVEL,
        PARENT_ROUTED_PARENT,
        "cpu" if CPU_ENABLE else None,
    )
    print(parent_routed_analysis_dir)


## 打包实验结果

In [ ]:
RESULT_ARCHIVE = PROJECT_ROOT / f"{EXPERIMENT_DIR.name}.zip"
PACKAGE_RESULT_ENABLE = True

if PACKAGE_RESULT_ENABLE:
    shutil.make_archive(str(RESULT_ARCHIVE.with_suffix("")), "zip", EXPERIMENT_DIR)
    print(RESULT_ARCHIVE)


In [ ]:
import zipfile

# 最后将当前数据集的 train 与 init 一起压缩到 data.zip。
DATA_ARCHIVE = PROJECT_ROOT / "data.zip"
DATA_SOURCE_DIRS = [
    dataset_dir / profile.root_train_clean,
    dataset_dir / profile.root_init,
]

with zipfile.ZipFile(DATA_ARCHIVE, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for source_dir in DATA_SOURCE_DIRS:
        if not source_dir.is_dir():
            raise FileNotFoundError(f"缺少待打包目录：{source_dir}")
        for file_path in source_dir.rglob("*"):
            if file_path.is_file():
                archive.write(file_path, file_path.relative_to(dataset_dir))

print(DATA_ARCHIVE)
